# Transforms and representations

Transforms return a new stream and never change the original. Every method is also a free function: `stream.flip_x()` is `ecv.flip_x(stream)`.

In [1]:
from pathlib import Path

import eventcv as ecv
import numpy as np


# Finds the small test recording in the EventCV repo. Set PATH to your own file instead.
def _fixture(rel="data/test/example.npz"):
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / rel).exists():
            return str(base / rel)
    raise FileNotFoundError(rel)


PATH = _fixture()

stream = ecv.load(PATH)
len(stream), stream.sensor_size

(106295, (640, 480))

## Transforms

Crop the sensor, then remove stuck pixels.

In [2]:
cropped = stream.crop(0, 0, 320, 240)  # x, y, width, height
clean = cropped.hot_pixel_filter()

print("events:", len(stream), "->", len(clean))
print("sensor size:", clean.sensor_size)

events: 106295 -> 19859
sensor size: (320, 240)


## Methods and free functions

Both forms give identical results.

In [3]:
flipped = stream.flip_x()
a = flipped.voxel(bins=5).numpy()
b = ecv.voxel(ecv.flip_x(stream), bins=5).numpy()
np.array_equal(a, b)

True

## Representations

Each representation returns an `EventFrame`; `.numpy()` gives the `(C, H, W)` array.

In [4]:
for name, frame in [
    ("count", clean.count()),
    ("voxel(bins=5)", clean.voxel(bins=5)),
    ("tsurf", clean.tsurf()),
]:
    arr = frame.numpy()
    print(f"{name:16} {str(arr.shape):18} {arr.dtype}")

count            (1, 240, 320)      uint64
voxel(bins=5)    (5, 240, 320)      float32
tsurf            (2, 240, 320)      float32


## As a PyTorch dataset

With `repr=`, a reader returns arrays: `ds[i]` is `(C, H, W)` and `ds.batch(indices)` is `(B, C, H, W)`.

In [5]:
ds = ecv.open(PATH, dt_ms=20, repr="count")
print("frames:", len(ds))
print("ds[0]:", ds[0].shape, ds[0].dtype)
print("batch([0, 1]):", ds.batch([0, 1]).shape)

frames: 3
ds[0]: (1, 480, 640) uint64
batch([0, 1]): (2, 1, 480, 640)


Pass the reader straight to a `DataLoader`:

```python
import torch

ds = ecv.open("recording.h5", dt_ms=30, repr="count")
loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=True)
```

Without `repr=`, items are streams of different lengths. Pass `collate_fn=ecv.collate` to batch them as lists. See [Large files](../large-files.md).